In [1]:
import shutil
shutil.unpack_archive('turkish_hate_model.zip', 'turkish_hate_model')

ReadError: turkish_hate_model.zip is not a zip file

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained('turkish_hate_model')
model = BertForSequenceClassification.from_pretrained('turkish_hate_model')
model.eval()

In [ ]:
import re

def metin_temizle(metin):
    metin = str(metin).lower()
    metin = re.sub(r'http\S+', '', metin)
    metin = re.sub(r'@\w+', '', metin)
    metin = re.sub(r'#\w+', '', metin)
    metin = re.sub(r'[^\w\s\u0080-\uffff]', '', metin)
    metin = re.sub(r'[^\u0000-\u05FF\s]', '', metin)
    metin = re.sub(r'\s+', ' ', metin)
    metin = metin.strip()
    return metin

In [ ]:
def tahmin_et(metin):
    metin_temiz = metin_temizle(metin)
    inputs = tokenizer(metin_temiz, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    etiket = "Nefret Söylemi" if pred == 1 else "Nötr"
    guven = probs[0][pred].item()
    return f"{etiket} (güven: {guven:.2%})"


print(tahmin_et("bu çok güzel bir gün"))
print(tahmin_et("örnek bir test cümlesi"))


In [ ]:
!pip install gradio -q

import gradio as gr

arayuz = gr.Interface(
    fn=tahmin_et,
    inputs=gr.Textbox(label="Metni girin", placeholder="Bir cümle yazın..."),
    outputs=gr.Textbox(label="Sonuç"),
    title="Türkçe Nefret Söylemi Tespiti"
)
arayuz.launch()

In [ ]:
from huggingface_hub import HfApi, create_repo

create_repo(
    repo_id="AtalhaG/turkish-hate-speech-model",
    repo_type="model",
    private=False
)


In [ ]:
from huggingface_hub import HfApi, login

login("")
api = HfApi()
api.upload_folder(
    folder_path="turkish_hate_model",
    repo_id="AtalhaG/turkish-hate-speech-model",
    repo_type="model"
)